In [ ]:
%load_ext autoreload
%autoreload 2
# import torch
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import pickle
from scipy.optimize import fmin_l_bfgs_b
import json
import os
from scipy.ndimage import gaussian_filter1d
import glob
from scipy.stats import norm
# import pandas as pd
# import seaborn as sns

# # Function to find the project root directory
# for parent in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents):
#     if (parent / "utils").exists():
#         sys.path.insert(0, str(parent))
#         break

# from utils.pathing import setup_repo_path
# ROOT = setup_repo_path()
# print(f"Project root set to: {ROOT}")
# -----------------------------------------------

# from utils.data import custom_optimizer

In [ ]:
# from limit_states import REGISTRY as ls_REGISTRY
# lstate = ls_REGISTRY['himmelblau']()
# # lstate = ls_REGISTRY['high_dimensional']()

# Pf_ref, B_ref, x_mc_physical, y_mc = lstate.monte_carlo_estimate(1e8)
# Pf_ref, B_ref
# x_mc_physical.mean(dim=0)
# x_mc_physical.std(dim=0)

Define the arguments from the experiment variations

In [ ]:
# --- Case study groups -------------------------------------------------------
group_2D = ['four_branch_6', 'four_branch_7', 'hat', 'himmelblau']
group_HD = ['nonlinear_oscillator', '2dof_oscillator', 'high_dimensional']

# Assign target lengths (initial 10 samples + AL iterations)
max_length = {
    case: 202 for case in group_2D
}
max_length.update({
    case: 502 for case in group_HD
})

# # Assign target Pf_model length ( including 10 DoE samples )
# max_length = {case: 202 for case in group_2D}
# max_length.update({case: 502 for case in group_HD})

In [ ]:
import os
import glob
import json

# --- Case study groups -------------------------------------------------------
group_2D = ['four_branch_6', 'four_branch_7', 'hat', 'himmelblau']
group_HD = ['nonlinear_oscillator', '2dof_oscillator', 'high_dimensional']

# Combined list (preserves original order)
casestudy = group_2D + group_HD

custom_titles = [
    r'Four-branch, $k=6$', r'Four-branch, $k=7$', 'Hat', 'Himmelblau',
    r'Nonlinear oscillator', '2-DOF Oscillator', r'High-dimensional'
]

real_pf_values = {
    'four_branch_6': 0.004458488011732697,
    'four_branch_7': 0.0022232679883018138,
    'hat': 0.00038667799963150175,
    'himmelblau': 1.65E-4,
    'nonlinear_oscillator': 0.0286178,
    '2dof_oscillator': 0.0047598,
    'high_dimensional': 0.0019820
}

# Base directory of results
# base_results_dir = '/Volumes/Jonathan/MOO_results/Results/results'
base_results_dir = r'D:\\active_train\\acquisition\\MOO-AL\\notebooks\\all_results\\results_tracking'

# Active learning settings
n_exp = 15
al_strategy = [
    'moo_reliability', 'moo_knee', 'moo_compromise',
    'moo_eps_greedy', 'eff', 'u', 'erf', 'reif', 'reif2', 'portfolio'
]
al_batch = [1]
name_exp = list(range(1, n_exp + 1))

# Dictionaries to store results
output_results_dict = {}
config_results_dict = {}

# --- Load all experiments ----------------------------------------------------
for case in casestudy:
    for strategy in al_strategy:
        for batch in al_batch:
            for exp_num in name_exp:

                dir_pattern = os.path.join(
                    base_results_dir, case, f"{strategy}_{batch}_{exp_num}_*"
                )

                matching_dirs = glob.glob(dir_pattern)

                if not matching_dirs:
                    print(f"No directory found for {case}, {strategy}, batch {batch}, exp {exp_num}")
                    continue

                key = f"{case}_{strategy}_{batch}_{exp_num}"

                # Load output.json
                output_json_path = os.path.join(matching_dirs[0], 'output.json')
                if os.path.isfile(output_json_path):
                    with open(output_json_path, 'r') as f:
                        output_results_dict[key] = json.load(f)
                else:
                    print(f"No output.json found for {key}")

                # Load config.json
                config_json_path = os.path.join(matching_dirs[0], 'config.json')
                if os.path.isfile(config_json_path):
                    with open(config_json_path, 'r') as f:
                        config_results_dict[key] = json.load(f)
                else:
                    print(f"No config.json found for {key}")


In [ ]:
incomplete_runs = []

for key, out in output_results_dict.items():
    # 1) Identify the case as the prefix of the key
    case = None
    for c in casestudy:
        if key.startswith(c + "_"):
            case = c
            break

    if case is None:
        print(f"Warning: could not identify case for key={key}")
        continue

    # 2) Remove the case + '_' prefix, then split the rest
    rest = key[len(case) + 1:]  # everything after 'case_'
    # rest should be 'strategy_batch_exp'
    try:
        strategy, batch, exp_num = rest.rsplit("_", 2)
    except ValueError:
        print(f"Warning: could not split strategy/batch/exp for key={key}")
        continue

    # 3) Now you can safely use `case` to look up max_length
    if 'Pf_model' not in out:
        incomplete_runs.append((key, "missing Pf_model"))
        continue

    pf_len = len(out['Pf_model'])
    target_len = max_length[case]-10  # 200 for 2D, 500 for HD

    if pf_len < target_len:
        incomplete_runs.append((key, pf_len))

print("\nIncomplete runs:")
for key, info in incomplete_runs:
    print(f"{key:50s} -> {info}")

In [ ]:
# name = 'updatedruns'
# with open(f'output_results_dict_{name}.pkl', 'wb') as fp:
#     pickle.dump(output_results_dict, fp)

# with open(f'config_results_dict_{name}.pkl', 'wb') as fp:
#     pickle.dump(config_results_dict, fp)

In [ ]:
# dir = '/Users/jonathan/Documents/MOAL/Experiments/AL_StructuralReliability/'
# dir = 'D:/active_train/compromised/AL_StructuralReliability/'

# name = 'pareto' #practical, analytical, pareto

# with open(dir+f'output_results_dict_{name}.pkl', 'rb') as fp:
#     loaded_file = pickle.load(fp)

# output_results_dict = loaded_file

# with open(dir+f'config_results_dict_{name}.pkl', 'rb') as fp:
#     loaded_file = pickle.load(fp)

# config_results_dict = loaded_file

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

sigma = 1.5  # smoothing

# Figure setup
fig, axs = plt.subplots(1, len(casestudy), figsize=(22, 5), sharey=False)
fig.suptitle(r"$P_f$ model evolution over experiments", x=0.5, fontsize=16)
fig.subplots_adjust(wspace=0.05, hspace=0.2)

# Strategy colors
strategy_colors = {
    'moo_reliability': "#1f77b4",
    'moo_knee': '#ff7f0e', 
    'moo_compromise': '#2ca02c',
    'moo_eps_greedy': '#8c564b',
    'erf': '#17becf', 
    'reif': "#432ca0",
    'reif2': '#e377c2',
    'eff': '#d62728',
    'u': '#9467bd',
    'portfolio': '#7f7f7f'
}

custom_legend = [r'MOO-R', r'MOO-K', r'MOO-C', r'MOO-EpsG',
                 r'ERF', r'REIF', r'REIF2', r'EFF', r'U']

# --- PLOTTING LOOP ------------------------------------------------------------
for i, case in enumerate(casestudy):

    ax = axs[i]
    ax.set_title(custom_titles[i])
    ax.set_xlabel("Training samples")

    if i == 0:
        ax.set_ylabel(r"$P_f$ model")

    for strategy in al_strategy:

        pf_models_all_experiments = []

        # load all 15 experiments for this case+strategy
        for exp_num in range(1, n_exp + 1):

            key = f"{case}_{strategy}_1_{exp_num}"

            if key not in output_results_dict:
                continue

            pf = output_results_dict[key].get("Pf_model", None)
            if pf is None:
                continue

            pf_models_all_experiments.append(np.array(pf, dtype=float))

        if not pf_models_all_experiments:
            continue

        # AUTOTRIM: keep only up to the shortest available run
        min_len = min(len(pf) for pf in pf_models_all_experiments)
        pf_trimmed = np.vstack([pf[:min_len] for pf in pf_models_all_experiments])

        # compute mean and std across experiments
        mean_pf = np.mean(pf_trimmed, axis=0)
        std_pf = np.std(pf_trimmed, axis=0)

        # smoothing
        mean_pf_s = gaussian_filter1d(mean_pf, sigma=sigma)
        std_pf_s = gaussian_filter1d(std_pf, sigma=sigma)

        # X-axis = sample index (10 initial samples included)
        steps = np.arange(min_len)

        ax.plot(
            steps, mean_pf_s,
            color=strategy_colors.get(strategy, "gray"),
            lw=1.5
        )

        ax.fill_between(
            steps,
            mean_pf_s - std_pf_s,
            mean_pf_s + std_pf_s,
            color=strategy_colors.get(strategy, "gray"),
            alpha=0.25
        )

    # Reference Pf line
    ax.axhline(y=real_pf_values[case], color='k', linestyle='--', linewidth=1.2)

    # Scales & limits --------------------------
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.grid(True, which="both", linewidth=0.2)

    # Different maximum sample lengths per group
    if case in group_2D:
        ax.set_xlim(10, 200)
    else:
        ax.set_xlim(10, 500)

# ---- Legend -----------------------------------------
handles = [
    plt.Line2D([0], [0], color=color, marker='s', linestyle='', markersize=10, label=leg)
    for leg, color in zip(custom_legend, strategy_colors.values())
]
handles.append(
    plt.Line2D([0], [0], color='k', linestyle='--', linewidth=1.6, label='Ref. $P_f$')
)

fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=10)

plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()


In [ ]:
Notebook: Cell Focus Indicator
Notebook: Diff Editor
Notebook: Cell Toolbar Visibility



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# labels in the desired order
strategies_order = ['moo_reliability','moo_knee', 'moo_compromise',
                    'moo_eps_greedy', 'erf', 'reif', 'reif2', 'eff', 'u', 'portfolio']
custom_legend   = [r'MOO-R', r'MOO-K', r'MOO-C', r'MOO-EpsG',
                   r'ERF', r'REIF', r'REIF2', r'EFF', r'U', r'Portfolio']

target_epsilon = {
    'four_branch_6':     [5e-3, 1e-2, 5e-2], 
    'four_branch_7':     [5e-3, 1e-2, 5e-2],
    'hat':               [2e-2, 5e-2, 1e-1],
    'himmelblau':        [2e-2, 5e-2, 1e-1],
    'nonlinear_oscillator': [2e-3, 5e-3, 1e-2],
    '2dof_oscillator':   [5e-2, 1e-2, 2e-1],
    'high_dimensional':  [5e-2, 3e-2, 1e-2]
}

font_size = 8
linewidth = 0.8
cm = 1/2.54  # centimeters in inches
sigma = 1.5  # for Gaussian smoothing
doe = 10     # initial DoE size

plt.rcParams.update({
    'font.size': font_size,
    'legend.fontsize': font_size,
    'legend.title_fontsize': font_size,
    'axes.titlesize': font_size,
    'axes.labelsize': font_size,
    'xtick.labelsize': font_size,
    'ytick.labelsize': font_size,
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'stix',
})

fig, axs = plt.subplots(
    1, len(casestudy),
    figsize=(19*cm, 4.0*cm),
    sharey=True, sharex=False
)

epsilon_color = '#4d4d4d'  # for target epsilon lines

for i, case in enumerate(casestudy):
    ax = axs[i]
    ax.set_title(f"{custom_titles[i]}", fontsize=font_size)

    if i == 0:
        ax.set_ylabel(r"$\delta P_\mathrm{f}$")

    reference_pf = float(real_pf_values[case])

    # Decide max length for this case (200 or 500)
    if case in group_2D:
        case_max_len = 200
    else:
        case_max_len = 500

    for strategy in al_strategy:
        # Skip strategies that are not in plotting order / color dict (e.g. 'portfolio')
        if strategy not in strategies_order or strategy not in strategy_colors:
            continue

        relative_diffs_all_exp = []

        for exp_num in range(1, n_exp + 1):
            key = f"{case}_{strategy}_1_{exp_num}"
            if key not in output_results_dict:
                continue

            pf_model = np.asarray(
                output_results_dict[key].get('Pf_model', []),
                dtype=float
            )

            if pf_model.size == 0:
                continue

            # Relative Pf error per iteration
            rel_diff = np.abs(pf_model - reference_pf) / reference_pf
            relative_diffs_all_exp.append(rel_diff)

        if not relative_diffs_all_exp:
            continue

        # --- Handle incomplete runs WITHOUT truncating complete ones -----------
        # We pad with NaNs and use nan-aware stats.
        max_len_available = max(len(diff) for diff in relative_diffs_all_exp)
        max_len = min(max_len_available, case_max_len)

        n_runs = len(relative_diffs_all_exp)
        rel_diff_mat = np.full((n_runs, max_len), np.nan, dtype=float)

        for r, diff in enumerate(relative_diffs_all_exp):
            this_len = min(len(diff), max_len)
            rel_diff_mat[r, :this_len] = diff[:this_len]

        # nan-aware statistics along experiments axis
        p2_5  = np.nanpercentile(rel_diff_mat,  2.5, axis=0)
        p50   = np.nanpercentile(rel_diff_mat, 50.0, axis=0)
        p97_5 = np.nanpercentile(rel_diff_mat, 97.5, axis=0)
        mean_rel = np.nanmean(rel_diff_mat, axis=0)

        # Gaussian smoothing
        p2_5_smooth  = gaussian_filter1d(p2_5,  sigma=sigma)
        p50_smooth   = gaussian_filter1d(p50,   sigma=sigma)
        p97_5_smooth = gaussian_filter1d(p97_5, sigma=sigma)
        mean_rel_smooth = gaussian_filter1d(mean_rel, sigma=sigma)

        # X-axis (number of acquired samples)
        steps = np.arange(doe, doe + max_len)

        pretty_label = custom_legend[strategies_order.index(strategy)]

        ax.plot(
            steps, p50_smooth,
            label=pretty_label,
            linestyle='solid',
            color=strategy_colors[strategy],
            linewidth=linewidth
        )
        ax.fill_between(
            steps, p2_5_smooth, p97_5_smooth,
            color=strategy_colors[strategy],
            alpha=0.1
        )

    # Target epsilon lines
    for eps in target_epsilon[case]:
        ax.axhline(
            y=eps,
            color=epsilon_color,
            linestyle='--',
            linewidth=0.5,
            alpha=0.8
        )

    ax.set_yscale('log')
    ax.set_ylim(1e-4, 1)
    ax.grid(True, which="both", linewidth=0.01, alpha=0.3)

# X-limits and ticks per case group
for ax, case in zip(axs, casestudy):
    if case in group_2D:
        ax.set_xlim(10, 200)
        ax.set_xticks([10, 100, 200])
    else:
        ax.set_xlim(10, 500)
        ax.set_xticks([10, 100, 500])
    ax.tick_params(width=0.3)

# --- Figure-level legend -----------------------------------------------------
handles = []
for strat_key, label in zip(strategies_order, custom_legend):
    handles.append(
        plt.Line2D(
            [0], [0],
            color=strategy_colors[strat_key],
            marker='s',
            markersize=5,
            linestyle='',
            label=label
        )
    )

handles.append(
    plt.Line2D(
        [0], [1],
        color=epsilon_color,
        linestyle='--',
        linewidth=0.8,
        label=r'$\delta P_{\mathrm{f,target}}$'
    )
)

fig.legend(
    handles=handles,
    loc="lower center",
    ncol=len(handles),
    fontsize=font_size,
    title_fontsize=font_size,
    columnspacing=1.0,
    handletextpad=0.5,
    handlelength=1.5,
    bbox_to_anchor=(0.5, -0.35, 0.0, 0.0),
)

fig.text(
    0.5, -0.07,
    "Number of acquired samples",
    ha='center',
    va='center',
    fontsize=font_size
)

fig.subplots_adjust(wspace=0.2, hspace=1.2)

for ax in axs:
    ax.tick_params(width=0.3, which='minor')
    ax.tick_params(width=0.3, which='major')
    for spine in ax.spines.values():
        spine.set_linewidth(0.3)

plt.show()

In [ ]:
fig.savefig(f'pf_evolution_.pdf',bbox_inches='tight')

In [ ]:
# casestudy, al_strategy
# case = casestudy[5]
# strategy = al_strategy[0]
# # for i, case in enumerate(casestudy):

# # for strategy in al_strategy:
#     # relative_diffs_all_exp = []
# for exp_num in range(1, n_exp + 1):
#     key = f"{case}_{strategy}_1_{exp_num}"
#     pf_evol_exp = output_results_dict[key].get('Pf_model', [])
#     plt.plot(pf_evol_exp)

# plt.axhline(y=real_pf_values[case], color=epsilon_color, linestyle='--', linewidth=1, alpha=0.8)

# plt.ylabel(f'Pf model')
# plt.xlabel(f'Training samples')

# # for i, case in enumerate(casestudy):
# reference_pf = real_pf_values[case]
# reference_B = calculate_reliability_index([reference_pf])[0]
# # for strategy in al_strategy:
#     # relative_diffs_all_exp = []
# for exp_num in range(1, n_exp + 1):
#     key = f"{case}_{strategy}_1_{exp_num}"
#     pf_evol_exp = output_results_dict[key].get('Pf_model', [])
#     b_index_evol = calculate_reliability_index(pf_evol_exp)
#     relative_diff = np.abs((b_index_evol - reference_B) / reference_B)
#     plt.plot(b_index_evol)

# plt.yscale('log')
# plt.ylabel(f'b_index_relative_diff')
# plt.xlabel(f'Training samples')

In [ ]:
# fig.savefig(f'relative_B_index_evolution.pdf',bbox_inches='tight')

In [ ]:
consecutive_iterations = 3

font_size = 8
linewidth = 0.3
cm = 1/2.54  # centimeters in inches
plt.rcParams.update({
    'font.size': font_size,
    'legend.fontsize': font_size,
    'legend.title_fontsize': font_size,
    'axes.titlesize': font_size,
    'axes.labelsize': font_size,
    'xtick.labelsize': font_size ,
    'ytick.labelsize': font_size ,
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'stix',
})

# Corrected plotting with case study titles at the top of each column
fig, axs = plt.subplots(3, len(casestudy), figsize=(19*cm, 13*cm), sharex=True, sharey=True)
# fig.suptitle(r'Number of training samples to meet $\delta\beta_\mathrm{target}$ in 3 consecutive iterations', y=0.95, fontsize=font_size)

data_per_case = {}
flierprops = dict(marker='o', markersize=3, markerfacecolor='white', linestyle='none', markeredgewidth=linewidth)
whiskerprops = dict(linewidth=linewidth, color='black')
capprops = dict(linewidth=linewidth, color='black')
# Initialize a dictionary to store unmet requirements per stability threshold
threshold_to_level = {
    case: { eps: lvl
            for eps, lvl in zip(eps_list, ['l','m','h']) }
    for case, eps_list in target_epsilon.items()
}
failures = {
    lvl: { strat: [0]*len(casestudy) for strat in al_strategy }
    for lvl in ('l','m','h')
}
# unmet_requirements = {}


# # Iterate over each case study and its corresponding thresholds
# for case, thresholds in target_epsilon.items():
#     for threshold in thresholds:
#         # Ensure the dictionary structure exists for the current threshold
#         if threshold not in unmet_requirements:
#             unmet_requirements[threshold] = {}
#         if case not in unmet_requirements[threshold]:
#             unmet_requirements[threshold][case] = {strategy: [] for strategy in al_strategy}
            
# Iterate over each case study and epsilon threshold
for i, case in enumerate(casestudy):
    reference_pf = real_pf_values[case]
    reference_B = calculate_reliability_index([reference_pf])[0]
    # print(f"Case: {case}")
    # Add case study title at the top of each column
    fig.text((i + 0.5) / len(casestudy), 0.94, f"{custom_titles[i]}", ha='center', va='center', fontsize=font_size)

    for j, stability_threshold in enumerate(target_epsilon[case]):
        # Reset min_training_samples_needed dictionary
        min_training_samples_needed = {strategy: [] for strategy in al_strategy}

        # Process each experiment and strategy
        for strategy in al_strategy:
            for exp_num in range(1, n_exp + 1):
                key = f"{case}_{strategy}_1_{exp_num}"
                if key in output_results_dict:
                    pf_model_evolution = output_results_dict[key].get('Pf_model', [])
                    doe = 10  # Simulated DoE
                    
                    # Calculate reliability index and relative difference
                    B_evolution = calculate_reliability_index(pf_model_evolution)
                    relative_diff = np.abs((B_evolution - reference_B) / reference_B)

                    # Find minimum training samples needed
                    consecutive_count = 0
                    for k, diff in enumerate(relative_diff):
                        if diff < stability_threshold:
                            consecutive_count += 1
                            if consecutive_count >= consecutive_iterations:
                                min_training_samples_needed[strategy].append((k + 1) + doe)
                                break
                        else:
                            consecutive_count = 0
                    else:
                        # If stability condition was never met, increment the counter
                        # unmet_requirements_counter[case][strategy] += 1
                        # unmet_requirements[stability_threshold][case][strategy].append(exp_num)
                        # If no stability was achieved, append a default value (e.g., 201)
                        lvl = threshold_to_level[case][stability_threshold]
                        idx = casestudy.index(case)            # picks the right slot
                        failures[lvl][strategy][idx] += 1  # bump the count

                        min_training_samples_needed[strategy].append(201)

        # Prepare data for box plot
        data_boxplot = [min_training_samples_needed[strategy] for strategy in al_strategy]

        strategies_reversed = al_strategy[::-1]
        data_boxplot.reverse()

        # Threshold label for the subplot
        ax = axs[j, i]
        mantissa = f"{stability_threshold:.0e}".split('e')[0]    # '1'
        exponent = int(f"{stability_threshold:.0e}".split('e')[1])  # -3
        ax.set_title (rf"$\delta\beta_\text{{target}} = {mantissa} \cdot 10^{{{exponent}}}$", fontsize=font_size, pad=3)
        # ax.set_title(rf"$\delta\beta$ = {stability_threshold:.0E}", fontsize=font_size, pad=3)

        # Plot box plots
        boxprops = dict(linewidth=linewidth)
        medianprops = dict(linewidth=0.5, color='k', ls='--')
        meanprops = dict(linewidth=0.5, color='k', ls='-')
        width = 0.5

        bplot = ax.boxplot(
            data_boxplot, vert=False, patch_artist=True, tick_labels=strategies_reversed, meanline=True, 
            showmeans=True, widths=width, boxprops=boxprops, medianprops=medianprops, meanprops=meanprops, whis=[2.5, 97.5], 
            flierprops=flierprops, whiskerprops=whiskerprops, capprops=capprops
        )

        # Apply colors to each box
        for patch, strategy in zip(bplot['boxes'], strategies_reversed):
            patch.set_facecolor(strategy_colors[strategy])

        ax.grid(True, which="both", linewidth=0.1, alpha=0.3)
        ax.set_xlim(10, 210)
        ax.set_xticks([10, 100, 201])
        ax.set_xticklabels(['10', '100', '>200'])          # only these three tick positions
        ax.tick_params(width=0.3)  
        ax.set_yticks([])

# Create custom legend
handles = [
    plt.Line2D([0], [0], color=color, marker='s', markersize=5, linestyle='', label=strategy) 
    for strategy, color in zip(custom_legend, strategy_colors.values())
]
handles.extend([
    plt.Line2D([0], [1], color='k', linestyle='--', linewidth=0.8, label='Median'),
    plt.Line2D([0], [1], color='k', linestyle='-', linewidth=0.8, label='Mean')
])

# Add legend
fig.legend(
    handles=handles, loc="lower center", ncol=len(handles), fontsize=font_size, title_fontsize=font_size,
    columnspacing=1.0, handletextpad=0.5, handlelength=1.5,bbox_to_anchor=(0.5, -0.02, 0.0, 0.0)
)
fig.text(
    0.5,           # x position (0=left, 0.5=center)
    0.05,          # y position (0=bottom, 1=top)
    "Number of acquired samples",
    ha='center',
    va='center',
    fontsize=font_size
)
plt.tight_layout(rect=[0, 0.05, 1.0, 0.94])

for ax in axs.flat:
    ax.tick_params(width=linewidth) 
    for spine in ax.spines.values():
        spine.set_linewidth(linewidth)

plt.show()

In [ ]:
fig.savefig(f'boxplots_epsilons.pdf',bbox_inches='tight')

In [ ]:
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

font_size = 8
linewidth = 0.3
cm = 1/2.54  # centimeters in inches
plt.rcParams.update({
    'font.size': font_size,
    'legend.fontsize': font_size,
    'legend.title_fontsize': font_size,
    'axes.titlesize': font_size,
    'axes.labelsize': font_size,
    'xtick.labelsize': font_size ,
    'ytick.labelsize': font_size ,
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'stix',
})
plt.rcParams.update({
    'axes.linewidth': 0.3,
    'xtick.major.width': 0.3,
    'ytick.major.width': 0.3,
    'lines.linewidth': 0.3,
})

case_studies = custom_titles
levels        = ['l','m','h']
# level_labels  = [r'$\delta\beta$ (l)', r'$\delta\beta$ (m)', r'$\delta\beta$ (h)']
level_labels = [r'$\delta\beta_\mathrm{l}$', r'$\delta\beta_\mathrm{m}$', r'$\delta\beta_\mathrm{h}$']

level_alphas  = [0.3, 0.6, 1.0]  # light → dark for l → h
# colors_base   = plt.cm.tab10.colors[:len(case_studies)]
colors_base = [
    "#e41a1c",  # blue
    "#4daf4a",  # orange
    "#984ea3",  # green
    "#a65628",  # red
    "#ff7f00",  # purple
    "#377eb8",  # brown
]

# 2) Build plot as before, but give bar edges an explicit thin line
fig, ax = plt.subplots(figsize=(9*cm, 6*cm))

x = np.arange(len(al_strategy))
bar_w = 0.6
bottom = np.zeros_like(x, dtype=float)

for cs_idx, cs_name in enumerate(case_studies):
    base_color = colors_base[cs_idx]
    for lvl_idx, lvl in enumerate(levels):
        heights = [failures[lvl][rs][cs_idx] for rs in al_strategy]
        color = to_rgba(base_color, alpha=level_alphas[lvl_idx])
        ax.bar(
            x, heights, bar_w, bottom=bottom,
            color=color,
            edgecolor='black',
            linewidth=0.2     # <--- thin bar border
        )
        bottom += heights

# 3) Thin out the remaining plot elements
for spine in ax.spines.values():
    spine.set_linewidth(0.2)
ax.tick_params(width=0.2, length=3)

# 6) Legend
case_patches = [
    Patch(facecolor=colors_base[i], edgecolor='black',
          label=case_studies[i])
    for i in range(len(case_studies))
]
level_patches = [
    Patch(facecolor='grey', edgecolor='black',
          alpha=level_alphas[i], label=level_labels[i])
    for i in range(len(levels))
]

# 4) Two separate legends
#   a) Case studies legend
legend1 = ax.legend(
    handles=case_patches,
    # title="Case studies",
    loc='upper left',
    bbox_to_anchor=(0.01, 0.99),
    fontsize=font_size*0.8,
    title_fontsize=font_size*0.9,
    handlelength=1.2,
    handletextpad=0.5,
)
ax.add_artist(legend1)

# #   b) Levels legend
# legend2 = ax.legend(
#     handles=level_patches,
#     # title=r"$\delta\beta$ levels",
#     loc='upper left',
#     bbox_to_anchor=(0.40, 0.99),
#     fontsize=font_size*0.8,
#     title_fontsize=font_size*0.9,
#     handlelength=1.2,
#     handletextpad=0.5,
# )

for legend in ([legend1]):
    # 1) Thin down the legend box itself
    legend.get_frame().set_linewidth(0.2)

    # 2) Thin down each handle
    for handle in legend.get_patches():
        handle.set_linewidth(0.1)
    for handle in legend.get_lines():
        handle.set_linewidth(0.1)

# 5) Rest of formatting
ax.set_xticks(x)
ax.set_xticklabels(custom_legend, rotation=0, ha='center')
ax.set_ylabel(r'Experiments exceeding $\delta\beta_\text{target}$ at $t=190$')


# ax.set_title('Agreggated number of failed experiments per strategy')
ax.grid(axis='y', linestyle='--', linewidth=0.2, alpha=0.5)
ax.set_ylim(0, 90)
ax.set_yticks(np.arange(0, 91, 10))

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig(f'Aggregated_N_failed_exp.pdf',bbox_inches='tight')

In [ ]:
# Stability parameters
stability_threshold = 0.04
consecutive_iterations = 3

# Initialize dictionary to store minimum training samples needed for each setting
min_training_samples_needed = {case: {strategy: [] for strategy in al_strategy} for case in casestudy}

# Loop through each casestudy and active learning strategy
for case in casestudy:
    reference_pf = real_pf_values[case]
    reference_B = calculate_reliability_index([reference_pf])[0]  # Calculate for reference as scalar
    
    for strategy in al_strategy:
        # Process each experiment
        for exp_num in range(1, n_exp + 1):
            # Construct key and retrieve Pf_model evolution data
            key = f"{case}_{strategy}_1_{exp_num}"
            if key in output_results_dict:
                pf_model_evolution = output_results_dict[key].get('Pf_model', [])
                doe = config_results_dict[key].get('doe')
                
                # Calculate reliability index for each Pf_model value in the evolution
                B_evolution = calculate_reliability_index(pf_model_evolution)
                
                # Calculate relative difference from reference reliability index
                relative_diff = np.abs((B_evolution - reference_B) / reference_B)
                
                # Check for the first point where stability requirements are met
                consecutive_count = 0
                for i, diff in enumerate(relative_diff):
                    if diff < stability_threshold:
                        consecutive_count += 1
                        if consecutive_count >= consecutive_iterations:
                            # Record the number of training samples needed and break
                            min_training_samples_needed[case][strategy].append((i + 1)+ doe)
                            break
                    else:
                        consecutive_count = 0
                else:
                    # If stability condition was never met, append a placeholder (e.g., NaN)
                    min_training_samples_needed[case][strategy].append(np.nan)

# Plotting the results
cm = 1/2.54  # centimeters in inches
plt.rcParams['font.family'] = 'times new roman'
plt.rcParams['font.size'] = 12
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'stix'
fig, axs = plt.subplots(1, len(casestudy), figsize=(35*cm, 10*cm), sharex=True)
fig.suptitle(rf'Number of training samples to reach $\epsilon$ < {stability_threshold:.1%} in {consecutive_iterations} consecutive iterations', y=0.95)

data_per_case = {}
for i, case in enumerate(casestudy):
    ax = axs[i]
    ax.set_title(f"{custom_titles[i]}")
    ax.set_xlabel("Training samples")
    if i == 0:
        ax.set_ylabel("Strategy")
    
    # Extract data for the current case study
    data_boxplot = [min_training_samples_needed[case][strategy] for strategy in al_strategy]

    print(f'Case Study: {case}, Data: {data_boxplot}')
    data_per_case[case] = data_boxplot
    
    # Reverse the order of data and strategies data.reverse()
    data_boxplot.reverse() 
    strategies_reversed = al_strategy[::-1]
    # Create horizontal box plots with different colors for each strategy
    median_color='k'
    mean_color='k'
    boxprops = dict(linewidth=1.0)
    medianprops = dict(linewidth=1.3, color=median_color, ls='--')
    meanprops= dict(linewidth=1.5, color=mean_color, ls='-')
    
    # Draw each box plot with specified color for each strategy
    width = 0.6
    bplot = ax.boxplot(
        data_boxplot, vert=False, patch_artist=True, tick_labels=strategies_reversed, meanline=True, showmeans=True, widths=width, 
        boxprops=boxprops, medianprops=medianprops, meanprops=meanprops, whis = [2.5,97.5]
    )
    
    # Apply colors to each box
    for patch, strategy in zip(bplot['boxes'], strategies_reversed):
        patch.set_facecolor(strategy_colors[strategy])
    
    ax.set_yticks([])
    ax.grid(True, which="both", linewidth = 0.2)
    common_xlim = (0, 200)
    # ax.set_xlim(0, max(plt.xlim()) + 10)  # Adjust x-limit for better visibility
    for ax in axs.flat:
        ax.set_xlim(common_xlim)

# Create custom handles for the strategy colors, median, and mean lines
handles = [
    plt.Line2D([0], [0], color=color, marker='s', markersize=10, linestyle='', label=strategy) 
    for strategy, color in zip(custom_legend, strategy_colors.values())
]
handles.append(plt.Line2D([0], [1], color=median_color, linestyle='--', linewidth=1.6, label='Median'))
handles.append(plt.Line2D([0], [1], color=mean_color, linestyle='-', linewidth=1.5, label='Mean'))

# Add a single legend at the bottom of the figure with strategy colors, median, and mean lines
fig.legend(handles=handles, title="Strategy and Statistics", loc="lower center", ncol=len(al_strategy) + 2)

plt.tight_layout(rect=[0, 0.13, 1, 1])
plt.show()

In [ ]:
# Stability parameters
stability_threshold = 0.04
consecutive_iterations = 3
custom_legend = [rf'MOO-R', rf'Knee', rf'Compromise', rf'EFF', rf'$U$']

# Function to calculate reliability index
def calculate_reliability_index(Pf):
    # Convert Pf to a NumPy array if it's a list or scalar
    Pf = np.array(Pf, dtype=np.float64)
    # Calculate reliability index, handling cases where Pf might be zero
    B = -norm.ppf(Pf)
    # Replace infinite values (from Pf=0) with a high reliability index value
    B[np.isinf(B)] = 10  # Assign a high value for infinite cases
    return B

# Initialize dictionary to store minimum training samples needed for each setting
min_training_samples_needed = {case: {strategy: [] for strategy in al_strategy} for case in casestudy}

# Loop through each casestudy and active learning strategy
for case in casestudy:
    reference_pf = real_pf_values[case]
    reference_B = calculate_reliability_index([reference_pf])[0]  # Calculate for reference as scalar
    
    for strategy in al_strategy:
        # Process each experiment
        for exp_num in range(1, n_exp + 1):
            # Construct key and retrieve Pf_model evolution data
            key = f"{case}_{strategy}_1_{exp_num}"
            if key in output_results_dict:
                pf_model_evolution = output_results_dict[key].get('Pf_model', [])
                doe = config_results_dict[key].get('doe')
                
                # Calculate reliability index for each Pf_model value in the evolution
                B_evolution = calculate_reliability_index(pf_model_evolution)
                
                # Calculate relative difference from reference reliability index
                relative_diff = np.abs((B_evolution - reference_B) / reference_B)
                
                # Check for the first point where stability requirements are met
                consecutive_count = 0
                for i, diff in enumerate(relative_diff):
                    if diff < stability_threshold:
                        consecutive_count += 1
                        if consecutive_count >= consecutive_iterations:
                            # Record the number of training samples needed and break
                            min_training_samples_needed[case][strategy].append((i + 1)+ doe)
                            break
                    else:
                        consecutive_count = 0
                else:
                    # If stability condition was never met, append a placeholder (e.g., NaN)
                    min_training_samples_needed[case][strategy].append(np.nan)

# Colors for each strategy (these will be used consistently across case studies)
strategy_colors = {
    'mo_reliability': '#1f77b4', 
    'knee': '#ff7f0e', 
    'compromise': '#2ca02c',
    'eff': '#d62728',
    'u': '#9467bd'
}

width = 0.6
# Plotting the results
cm = 1/2.54  # centimeters in inches
plt.rcParams['font.family'] = 'times new roman'
plt.rcParams['font.size'] = 12
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'stix'
fig, axs = plt.subplots(1, len(casestudy), figsize=(35*cm, 10*cm), sharex=True)
fig.suptitle(rf'Number of training samples to reach $\epsilon$ < {stability_threshold:.1%} in {consecutive_iterations} consecutive iterations', y=0.95)

data_per_case = {}
for i, case in enumerate(casestudy):
    ax = axs[i]
    ax.set_title(f"{custom_titles[i]}")
    ax.set_xlabel("Training samples")
    if i == 0:
        ax.set_ylabel("Strategy")
    
    # Extract data for the current case study
    data_boxplot = [min_training_samples_needed[case][strategy] for strategy in al_strategy]

    print(f'Case Study: {case}, Data: {data_boxplot}')
    data_per_case[case] = data_boxplot
    
    # Reverse the order of data and strategies data.reverse()
    data_boxplot.reverse() 
    strategies_reversed = al_strategy[::-1]
    # Create horizontal box plots with different colors for each strategy
    median_color='k'
    mean_color='k'
    boxprops = dict(linewidth=1.0)
    medianprops = dict(linewidth=1.3, color=median_color, ls='--')
    meanprops= dict(linewidth=1.5, color=mean_color, ls='-')
    
    # Draw each box plot with specified color for each strategy
    bplot = ax.boxplot(
        data_boxplot, vert=False, patch_artist=True, tick_labels=strategies_reversed, meanline=True, showmeans=True, widths=width, 
        boxprops=boxprops, medianprops=medianprops, meanprops=meanprops, whis = [2.5,97.5]
    )
    
    # Apply colors to each box
    for patch, strategy in zip(bplot['boxes'], strategies_reversed):
        patch.set_facecolor(strategy_colors[strategy])
    
    ax.set_yticks([])
    ax.grid(True, which="both", linewidth = 0.2)
    common_xlim = (0, 200)
    # ax.set_xlim(0, max(plt.xlim()) + 10)  # Adjust x-limit for better visibility
    for ax in axs.flat:
        ax.set_xlim(common_xlim)

# Create custom handles for the strategy colors, median, and mean lines
handles = [
    plt.Line2D([0], [0], color=color, marker='s', markersize=10, linestyle='', label=strategy) 
    for strategy, color in zip(custom_legend, strategy_colors.values())
]
handles.append(plt.Line2D([0], [1], color=median_color, linestyle='--', linewidth=1.6, label='Median'))
handles.append(plt.Line2D([0], [1], color=mean_color, linestyle='-', linewidth=1.5, label='Mean'))

# Add a single legend at the bottom of the figure with strategy colors, median, and mean lines
fig.legend(handles=handles, title="Strategy and Statistics", loc="lower center", ncol=len(al_strategy) + 2)

plt.tight_layout(rect=[0, 0.13, 1, 1])
plt.show()

In [ ]:
# fig.savefig(f'stab_{stability_threshold}_it_{consecutive_iterations}.pdf',bbox_inches='tight')

In [ ]:
# Initialize a dictionary to store data
data = {case: {strategy: {'selected_samples_runs': []} 
               for strategy in al_strategy} for case in casestudy}

for case in casestudy:
    for strategy in al_strategy:
        for exp_num in range(1, n_exp + 1):
            key = f"{case}_{strategy}_1_{exp_num}"
            if key in output_results_dict:
                pareto_metrics = output_results_dict[key].get('Pareto_metrics', [])
                selected_samples = []
                # print(f"Key: {key}, Pareto Metrics: {pareto_metrics}")
                for iteration_metrics in pareto_metrics:
                    if iteration_metrics:
                        selected_sample = iteration_metrics[2]
                        # selected_sample[0] = 1 - selected_sample[0]
                        selected_samples.append(selected_sample)
                data[case][strategy]['selected_samples_runs'].append(selected_samples)

In [ ]:
# def compute_intervals(sample, resolution=1_00_001):
#     F = np.column_stack(sample)
#     z_star = np.array([1.0, 1.0])
#     W = np.linspace(0, 1, resolution)
#     dev = np.abs(F - z_star)
#     T = np.zeros((F.shape[0], W.size))
#     for j, w in enumerate(W):
#         T[:, j] = np.sqrt((w * dev[:, 0]) ** 2 + ((1 - w) * dev[:, 1]) ** 2)
#     best = np.argmin(T, axis=0)
#     intervals = {}
#     current = best[0]
#     start = 0
#     for j in range(1, len(W)):
#         if best[j] != current:
#             intervals.setdefault(current, []).append((W[start], W[j - 1]))
#             current = best[j]
#             start = j
#     intervals.setdefault(current, []).append((W[start], W[-1]))
#     return [iv for ivs in intervals.values() for iv in ivs]

def compute_intervals(sample, resolution=100_00_001):
    # sample = [xx, yy]
    F = np.column_stack(sample)

    # ── 2) utopian point
    # z_star = F.max(axis=0)   # [max f1, max f2]
    z_star = np.array([1.0, 1.0])  # [max f1, max f2] 

    # ── 3) weight grid
    W = np.linspace(0, 1, resolution)   # 0.001 resolution

    # ── 4) compute Tchebycheff values: shape (n_points, n_weights)
    dev = np.abs(F - z_star)      # shape (15,2)
    # For each w: T_i(w) = max(w*dev[i,0], (1-w)*dev[i,1])
    T = np.maximum.outer(dev[:,0], W)        * 0  # placeholder
    # easier:
    T = np.zeros((F.shape[0], W.size))
    for j, w in enumerate(W):
        # T[:, j] = np.maximum(w*dev[:,0], (1-w)*dev[:,1])
        # new (Euclidean compromise)
        T[:, j] = np.sqrt( (w*dev[:,0])**2 + ((1-w)*dev[:,1])**2 )

    # ── 5) find the minimiser at each w
    best = np.argmin(T, axis=0)   # length = len(W)

    # ── 6) extract intervals
    intervals = {}
    current = best[0]
    start = 0
    for j in range(1, len(W)):
        if best[j] != current:
            intervals.setdefault(current, []).append((W[start], W[j-1]))
            current = best[j]
            start = j
    # close last run
    intervals.setdefault(current, []).append((W[start], W[-1]))

    # ── 7) print
    for idx, ivs in intervals.items():
        for lo, hi in ivs:
            # print(f"Point {idx:2d}: w ∈ [{lo:.6f}, {hi:.6f}]")
            return [(lo, hi)]
# for case in casestudy:
#     for strategy in al_strategy:
#         runs = data[case][strategy]['selected_samples_runs']
#         for run_id, run_samples in enumerate(runs):  # each run = list of 192 [x, y] samples
#             run_intervals = []
#             for i, point in enumerate(run_samples):
#                 try:
#                     interval = compute_intervals([point])  # Wrap single point in a list
#                     run_intervals.append(interval[0])  # Only one interval: full [0, 1]
#                 except Exception as e:
#                     print(f"❌ Error in {case} - {strategy} run {run_id}, point {i}: {e}")
#                     run_intervals.append(None)
#             intervals_data[case][strategy].append(run_intervals)


In [ ]:
run = data['four_branch_6']['mo_reliability']['selected_samples_runs'][0]

In [ ]:
run

In [ ]:
run[10][0] = 1 - run[10][0]

In [ ]:
compute_intervals(run[10], resolution=100_00_001) # Example data

In [ ]:
run[10]

In [ ]:
run[:10]

In [ ]:
# ── Initialize output structure
intervals_data = {
    case: {
        strategy: []
        for strategy in al_strategy
    }
    for case in casestudy
}

case = 'four_branch_6'
strategy = 'mo_reliability'
run = data[case][strategy]['selected_samples_runs'][0]
run_intervals = []
for i, point in enumerate(run[:5]):
    try:
        interval = compute_intervals([point], resolution=100_00_001)  # Wrap single point in a list
        print(interval)
        run_intervals.append(interval[0])  # Only one interval: full [0, 1]
    except Exception as e:
        print(f"❌ Error in {case} - {strategy} run {run_id}, point {i}: {e}")
        break
        run_intervals.append(None)
intervals_data[case][strategy].append(run_intervals)

In [ ]:
intervals_data[case][strategy]

In [ ]:
compute_intervals([point], resolution=10_00_001)[0]

Data for evolutions exploration-exploitation

In [ ]:
from statistics import mean, stdev
import math

# Initialize a dictionary to store data
data = {case: {strategy: {'selected_samples_runs': [], 'optimal_exploitation_runs': [], 'optimal_exploration_runs': []} 
               for strategy in al_strategy} for case in casestudy}

for case in casestudy:
    for strategy in al_strategy:
        for exp_num in range(1, n_exp + 1):
            key = f"{case}_{strategy}_1_{exp_num}"
            if key in output_results_dict:
                pareto_metrics = output_results_dict[key].get('Pareto_metrics', [])
                selected_samples = []
                optimal_exploitation = []
                optimal_exploration = []
                for iteration_metrics in pareto_metrics:
                    if iteration_metrics:
                        # Optimal exploitation objective
                        optimal_exploitation.append(iteration_metrics[0])
                        # Optimal exploration objective
                        optimal_exploration.append(iteration_metrics[1])
                        # Selected sample (the one chosen by the strategy)
                        selected_samples.append(iteration_metrics[2])

                        if math.isnan(iteration_metrics[2][0]):
                            print(case, strategy, exp_num, "contains NaN in selected samples")
                            
                data[case][strategy]['selected_samples_runs'].append(selected_samples)
                data[case][strategy]['optimal_exploitation_runs'].append(optimal_exploitation)
                data[case][strategy]['optimal_exploration_runs'].append(optimal_exploration)
                
stats = {case: {strategy: {
    'mean_exploration': [], 'mean_exploitation': [], 'std_exploration': [], 'std_exploitation': [],
    'mean_exploration_opt_exploit': [], 'mean_exploitation_opt_exploit': [],
    'mean_exploration_opt_explore': [], 'mean_exploitation_opt_explore': []
} for strategy in al_strategy} for case in casestudy}

for case in casestudy:
    for strategy in al_strategy:
        selected_runs = data[case][strategy]['selected_samples_runs']
        opt_exploit_runs = data[case][strategy]['optimal_exploitation_runs']
        opt_explore_runs = data[case][strategy]['optimal_exploration_runs']
        num_iterations = len(selected_runs[0])
        for i in range(num_iterations):
            # Selected samples
            exploration_values = [abs(run[i][0]) for run in selected_runs]
            exploitation_values = [1 -run[i][1] for run in selected_runs]
            # Filter out any NaN values
            exploration_values = [val for val in exploration_values if not math.isnan(val)]
            exploitation_values = [val for val in exploitation_values if not math.isnan(val)]
            # Calculate mean and standard deviation
            stats[case][strategy]['mean_exploration'].append(mean(exploration_values))
            stats[case][strategy]['mean_exploitation'].append(mean(exploitation_values))
            stats[case][strategy]['std_exploration'].append(stdev(exploration_values))
            stats[case][strategy]['std_exploitation'].append(stdev(exploitation_values))
            # # Optimal exploitation objective
            # opt_exploit_exploration = [abs(run[i][0]) for run in opt_exploit_runs]
            # opt_exploit_exploitation = [run[i][1] for run in opt_exploit_runs]
            # stats[case][strategy]['mean_exploration_opt_exploit'].append(mean(opt_exploit_exploration))
            # stats[case][strategy]['mean_exploitation_opt_exploit'].append(mean(opt_exploit_exploitation))
            # # Optimal exploration objective
            # opt_explore_exploration = [abs(run[i][0]) for run in opt_explore_runs]
            # opt_explore_exploitation = [run[i][1] for run in opt_explore_runs]
            # stats[case][strategy]['mean_exploration_opt_explore'].append(mean(opt_explore_exploration))
            # stats[case][strategy]['mean_exploitation_opt_explore'].append(mean(opt_explore_exploitation))


In [ ]:
import math
math.isnan(data[case][strategy]['selected_samples_runs'][0][0][0])

Plooting each seed

In [ ]:
import matplotlib.lines as mlines
from scipy.ndimage import gaussian_filter1d

# Custom strategy colors

strategy_colors = {
    'mo_reliability': '#1f77b4', 
    'knee': '#ff7f0e', 
    'compromise': '#2ca02c',
    'eff': '#d62728',
    'u': '#9467bd'
}

# Set up font and figure parameters
font_size = 8
linewidth = 0.8
cm = 1/2.54  # centimeters in inches
plt.rcParams.update({
    'font.size': font_size,
    'legend.fontsize': font_size,
    'legend.title_fontsize': font_size,
    'axes.titlesize': font_size,
    'axes.labelsize': font_size,
    'xtick.labelsize': font_size ,
    'ytick.labelsize': font_size ,
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'stix',
})

fig, axs = plt.subplots(2, len(casestudy), figsize=(19*cm, 7*cm), sharex=True, sharey='row')
# fig.suptitle(rf'Evolution of exploration-exploitation metrics', y=0.95, fontsize=font_size)
# Iterate over case studies and plot
for i, case in enumerate(casestudy):
    for strategy in al_strategy:
        # Retrieve number of iterations
        num_iterations = len(stats[case][strategy]['mean_exploration'])
        # X values represent training data sizes from 10 to (10 + num_iterations - 1)
        x_values = np.arange(10, 10 + num_iterations)

        # Individual runs
        selected_runs = data[case][strategy]['selected_samples_runs']
        color = strategy_colors[strategy]

        # Plot individual runs
        for run in selected_runs:
            exploration_values = [abs(sample[0]) for sample in run]
            exploitation_values = [1 - sample[1] for sample in run]
            axs[0, i].plot(x_values, exploration_values, color=color, linestyle='solid', alpha=0.03)
            axs[1, i].plot(x_values, exploitation_values, color=color, linestyle='solid', alpha=0.03)

        # Smooth the mean values using Gaussian filtering
        mean_exploration = np.array(stats[case][strategy]['mean_exploration'])
        mean_exploitation = np.array(stats[case][strategy]['mean_exploitation'])
        # print(f'{case} - {strategy} - Exploration: {mean_exploration}, Exploitation: {mean_exploitation}')
        sigma = 2.0  # Smoothing parameter
        mean_exploration_smooth = gaussian_filter1d(mean_exploration, sigma=sigma)
        mean_exploitation_smooth = gaussian_filter1d(mean_exploitation, sigma=sigma)

        # Plot the smoothed mean lines
        axs[0, i].plot(x_values, mean_exploration_smooth, color=color, linestyle='solid', linewidth=linewidth)
        axs[1, i].plot(x_values, mean_exploitation_smooth, color=color, linestyle='solid', linewidth=linewidth)

    # Set subplot titles for each case study
    axs[0, i].set_title(custom_titles[i])
    
for ax_row in axs:
    for ax in ax_row:
        ax.grid(True, which="both", linewidth=0.1)

common_xlim = (10, 200)
for ax in axs.flat:
    ax.set_xlim(common_xlim)

# Set shared labels
# Exploration metric label on the left row
axs[0, 0].set_ylabel("Exploration")
# Exploitation metric label on the left row
axs[1, 0].set_ylabel("Exploitation")

# Adjust spacing to make subplots closer
plt.tight_layout()

# Create custom legend
handles_strategies = [
    plt.Line2D([0], [0], color=color, marker='s', markersize=5, linestyle='', label=legend_label)
    for (strategy, color), legend_label in zip(strategy_colors.items(), custom_legend)]

handles_strategies.extend([
    plt.Line2D([0], [1], color='k', linestyle='-', linewidth=1.0, label='Median'),
    plt.Line2D([0], [1], color='k', linestyle='-', linewidth=0.4, alpha=0.2, label='Individual runs (shaded)')
])

# Add legend
fig.legend(
    handles=handles_strategies, loc="lower center", ncol=len(handles_strategies), 
    fontsize=font_size, title_fontsize=font_size, columnspacing=1.0, handletextpad=0.5, handlelength=1.5, bbox_to_anchor=(0.5, -0.06)
)
fig.text(
    0.5,           # x position (0=left, 0.5=center)
    0.08,          # y position (0=bottom, 1=top)
    "Number of acquired samples (active samples)",
    ha='center',
    va='center',
    fontsize=font_size
)
# Adjust the bottom space to fit the legend
plt.subplots_adjust(bottom=0.20, wspace=0.08, hspace=0.1)

for ax in axs.flat:
    ax.tick_params(width=0.3) 
    for spine in ax.spines.values():
        spine.set_linewidth(0.3)

plt.show()

In [ ]:
selected_runs

In [ ]:
# fig.savefig(f'exploration_exploitaion_evol_test.pdf',bbox_inches='tight')

In [ ]:
def compute_intervals(sample, resolution=100_00_001):
    # sample = [xx, yy]
    F = np.column_stack(sample)

    # ── 2) utopian point
    # z_star = F.max(axis=0)   # [max f1, max f2]
    z_star = np.array([1.0, 1.0])  # [max f1, max f2] 

    # ── 3) weight grid
    W = np.linspace(0, 1, resolution)   # 0.001 resolution

    # ── 4) compute Tchebycheff values: shape (n_points, n_weights)
    dev = np.abs(F - z_star)      # shape (15,2)
    # For each w: T_i(w) = max(w*dev[i,0], (1-w)*dev[i,1])
    T = np.maximum.outer(dev[:,0], W)        * 0  # placeholder
    # easier:
    T = np.zeros((F.shape[0], W.size))
    for j, w in enumerate(W):
        # T[:, j] = np.maximum(w*dev[:,0], (1-w)*dev[:,1])
        # new (Euclidean compromise)
        T[:, j] = np.sqrt( (w*dev[:,0])**2 + ((1-w)*dev[:,1])**2 )

    # ── 5) find the minimiser at each w
    best = np.argmin(T, axis=0)   # length = len(W)

    # ── 6) extract intervals
    intervals = {}
    current = best[0]
    start = 0
    for j in range(1, len(W)):
        if best[j] != current:
            intervals.setdefault(current, []).append((W[start], W[j-1]))
            current = best[j]
            start = j
    # close last run
    intervals.setdefault(current, []).append((W[start], W[-1]))

    # ── 7) print
    for idx, ivs in intervals.items():
        for lo, hi in ivs:
            # print(f"Point {idx:2d}: w ∈ [{lo:.6f}, {hi:.6f}]")
            return [(lo, hi)]

In [ ]:
selected_samples[2]

In [ ]:
selected_samples[2][0] = 1 - selected_samples[2][0]
# selected_samples[2][1] remains unchanged

In [ ]:
interval = compute_intervals(selected_samples[2], resolution=1_00_001)
interval

In [ ]:
import numpy as np

def compute_intervals(sample, resolution=5_00_001):
    F = np.column_stack(sample)
    z_star = np.array([1.0, 1.0])
    W = np.linspace(0, 1, resolution)
    dev = np.abs(F - z_star)
    T = np.zeros((F.shape[0], W.size))
    for j, w in enumerate(W):
        T[:, j] = np.sqrt((w * dev[:, 0]) ** 2 + ((1 - w) * dev[:, 1]) ** 2)
    best = np.argmin(T, axis=0)
    intervals = {}
    current = best[0]
    start = 0
    for j in range(1, len(W)):
        if best[j] != current:
            intervals.setdefault(current, []).append((W[start], W[j - 1]))
            current = best[j]
            start = j
    intervals.setdefault(current, []).append((W[start], W[-1]))
    # Return all intervals (may be multiple per point)
    return [iv for ivs in intervals.values() for iv in ivs]

# ── 1) Initialize output container
intervals_data = {
    case: {
        strategy: []
        for strategy in al_strategy
    }
    for case in casestudy
}

# ── 2) Extract intervals from selected samples
for case in casestudy:
    for strategy in al_strategy:
        selected_runs = data[case][strategy]['selected_samples_runs']
        for run_samples in selected_runs:  # 15 runs
            if all(isinstance(s, (list, tuple)) and len(s) == 2 for s in run_samples):
                try:
                    interval = compute_intervals(run_samples)
                    intervals_data[case][strategy].append(interval)
                except Exception as e:
                    print(f"Error computing interval for {case} - {strategy}: {e}")
            else:
                print(f"Invalid format in selected samples for {case} - {strategy}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Example lists/variables assumed to be defined previously:
# casestudy = [...]
# al_strategy = [...]
# data = {case: {strategy: {'selected_samples_runs': [...]}}}
# where each entry in 'selected_samples_runs' is a list of iterations, and each iteration is [exploration, exploitation]
'''KDE doesnt just map each data point onto a discrete location; it smooths the distribution by placing a kernel (often a Gaussian) centered at each data point and then sums (or averages) them to form a continuous probability density function'''

# Set up font and figure parameters
cm = 1/2.54  # centimeters in inches
plt.rcParams['font.family'] = 'times new roman'
plt.rcParams['font.size'] = 12
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'stix'

# Create the figure with one row and one column per case study
fig, axs = plt.subplots(1, len(casestudy), figsize=(35*cm, 12*cm), sharey=True)

# If only one case study, axs might not be a list
if len(casestudy) == 1:
    axs = [axs]

# Get a color palette for the strategies
colors = [strategy_colors[strat] for strat in al_strategy]

for i, case in enumerate(casestudy):
    ax = axs[i]
    for j, strategy in enumerate(al_strategy):
        # Gather all exploration and exploitation values from all runs
        runs = data[case][strategy]['selected_samples_runs']
        exploration_values = []
        exploitation_values = []
        for run in runs:
            for sample in run:
                exploration_values.append(abs(sample[0]))
                exploitation_values.append(1 - sample[1])

        # Plot individual points with low alpha
        # ax.scatter(exploration_values, exploitation_values, alpha=0.1, color=colors[j], s=5)
        
        # Overlay the KDE contours
        sns.kdeplot(
            x=exploration_values, 
            y=exploitation_values, 
            levels=20, 
            color=colors[j], 
            linewidths=0.5,
            ax=ax,
            cut=0,
            alpha=0.8
        )

    # Set titles and labels
    ax.set_title(custom_titles[i])
    ax.set_xlabel("Exploration")
    ax.grid(True)
    if i == 0:
        ax.set_ylabel("Exploitation")

    common_xlim = (0, 1)
    ax.set_xlim(common_xlim)

# Create a custom legend for strategies at the bottom of the figure
handles_strategies = [
    plt.Line2D([0], [0], color=color, marker='s', markersize=6, linestyle='', label=legend_label)
    for (strategy, color), legend_label in zip(strategy_colors.items(), custom_legend)
]

fig.legend(
    handles=handles_strategies, 
    title="Strategies", 
    loc="lower center", 
    ncol=len(al_strategy),
    bbox_to_anchor=(0.5, -0.05)
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.2, wspace=0.1)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Example lists/variables assumed to be defined previously:
# casestudy = [...]
# al_strategy = [...]
# data = {case: {strategy: {'selected_samples_runs': [...]}}}
# where each entry in 'selected_samples_runs' is a list of iterations, and each iteration is [exploration, exploitation]
'''KDE doesnt just map each data point onto a discrete location; it smooths the distribution by placing a kernel (often a Gaussian) centered at each data point and then sums (or averages) them to form a continuous probability density function'''

# Set up font and figure parameters
cm = 1/2.54  # centimeters in inches
plt.rcParams['font.family'] = 'times new roman'
plt.rcParams['font.size'] = 12
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'stix'

# Create the figure with one row and one column per case study
fig, axs = plt.subplots(1, len(casestudy), figsize=(35*cm, 12*cm), sharey=True)

# If only one case study, axs might not be a list
if len(casestudy) == 1:
    axs = [axs]

# Get a color palette for the strategies
colors = [strategy_colors[strat] for strat in al_strategy]

for i, case in enumerate(casestudy):
    ax = axs[i]
    for j, strategy in enumerate(al_strategy):
        # Gather all exploration and exploitation values from all runs
        runs = data[case][strategy]['selected_samples_runs']
        exploration_values = []
        exploitation_values = []
        for run in runs:
            for sample in run:
                exploration_values.append(abs(sample[0]))
                exploitation_values.append(1 - sample[1])

        # Plot individual points with low alpha
        ax.scatter(exploration_values, exploitation_values, alpha=0.1, color=colors[j], s=5)
        
        # Overlay the KDE contours
        sns.kdeplot(
            x=exploration_values, 
            y=exploitation_values, 
            levels=20, 
            color=colors[j], 
            linewidths=0.5,
            ax=ax,
            cut=0,
            alpha=0.8
        )

    # Set titles and labels
    ax.set_title(custom_titles[i])
    ax.set_xlabel("Exploration")
    ax.grid(True)
    if i == 0:
        ax.set_ylabel("Exploitation")

    common_xlim = (0, 1)
    ax.set_xlim(common_xlim)

# Create a custom legend for strategies at the bottom of the figure
handles_strategies = [
    plt.Line2D([0], [0], color=color, marker='s', markersize=6, linestyle='', label=legend_label)
    for (strategy, color), legend_label in zip(strategy_colors.items(), custom_legend)
]

fig.legend(
    handles=handles_strategies, 
    title="Strategies", 
    loc="lower center", 
    ncol=len(al_strategy),
    bbox_to_anchor=(0.5, -0.05)
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.2, wspace=0.1)
plt.show()

Practical examples

In [ ]:
# # Practical examples --------------------------------------------------------------
# custom_titles = [rf'High dimensional', rf'Nonlinear oscillator']
# casestudy = ['high_dimensional', 'nonlinear_oscillator']
# real_pf_values = {
#     'high_dimensional': 0.002003,
#     'nonlinear_oscillator': 0.0286178
# }
# base = 'results_pract'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Example case studies and strategies as previously defined
# casestudy = ['four_branch_6', 'four_branch_7', 'hat', 'himmelblau']
# al_strategy = ['knee', 'compromise', 'eff', 'u', 'mo_reliability']

# We'll use the default matplotlib color cycle again
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Prepare the figure: 1 row by number_of_case_studies columns
fig, axs = plt.subplots(1, len(casestudy), figsize=(14, 4), sharex=True, sharey=True)

# Iterate over each case study and subplot
for i, case in enumerate(casestudy):
    ax = axs[i]
    ax.set_title(case)

    # Collect handles for legend
    # We'll plot each strategy's density and mean line
    for idx, strategy in enumerate(al_strategy):
        # Extract all points from all runs for this case and strategy
        runs = data[case][strategy]['selected_samples_runs']
        all_points = []
        for run in runs:
            # run is a list of (exploration, exploitation) pairs
            for (explore_val, exploit_val) in run:
                all_points.append((explore_val, exploit_val))
        
        all_points = np.array(all_points)
        if all_points.size == 0:
            # If no points, skip
            continue

        x = all_points[:, 0]
        y = all_points[:, 1]

        # Plot a 2D density (KDE) of where the strategy tends to be
        # We'll use some transparency to allow overlapping densities to be visible
        sns.kdeplot(
            x=x, y=y, ax=ax, 
            shade=True, shade_lowest=False, 
            alpha=0.3, # transparency
            cmap="Blues" if idx == 0 else None, # You can vary colormaps if you like
            color=colors[idx],
            levels=5, # number of contour levels
            linewidths=1
        )

        # Now overlay the mean trajectory
        mean_exploration = stats[case][strategy]['mean_exploration']
        mean_exploitation = stats[case][strategy]['mean_exploitation']
        ax.plot(mean_exploration, mean_exploitation, marker='o', color=colors[idx], label=strategy, linewidth=2)

    # Set labels for the left and bottom plots only (since we share axes)
    if i == 0:
        ax.set_ylabel("Exploitation (Maximize)")
    ax.set_xlabel("Exploration (Maximize)")

# Adjust spacing and add a single legend at the bottom of the figure
handles, labels = axs[-1].get_legend_handles_labels()
fig.legend(handles, labels, title="Strategies", loc='lower center', ncol=len(al_strategy), bbox_to_anchor=(0.5, -0.2))

plt.tight_layout()
plt.subplots_adjust(bottom=0.25)  # space for the legend
plt.show()


In [ ]:
# Function to calculate reliability index
def calculate_reliability_index(Pf):
    Pf = np.array(Pf, dtype=np.float64)
    B = -norm.ppf(Pf)
    B[np.isinf(B)] = 10  # Assign a high value for infinite cases
    return B

# Function to calculate min_training_samples_needed based on stability criteria
def calculate_min_training_samples(casestudy, al_strategy, stability_threshold, consecutive_iterations):
    n_exp = 15  # Number of experiments
    
    min_training_samples_needed = {case: {strategy: [] for strategy in al_strategy} for case in casestudy}
    for case in casestudy:
        reference_pf = real_pf_values[case]
        reference_B = calculate_reliability_index([reference_pf])[0]
        
        for strategy in al_strategy:
            for exp_num in range(1, n_exp + 1):
                key = f"{case}_{strategy}_1_{exp_num}"
                if key in output_results_dict:
                    pf_model_evolution = output_results_dict[key].get('Pf_model', [])
                    doe = config_results_dict[key].get('doe')
                    B_evolution = calculate_reliability_index(pf_model_evolution)
                    relative_diff = np.abs((B_evolution - reference_B) / reference_B)
                    
                    consecutive_count = 0
                    for i, diff in enumerate(relative_diff):
                        if diff < stability_threshold:
                            consecutive_count += 1
                            if consecutive_count >= consecutive_iterations:
                                min_training_samples_needed[case][strategy].append((i + 1) + doe) #including initial DoE
                                break
                        else:
                            consecutive_count = 0
                    else:
                        min_training_samples_needed[case][strategy].append(np.nan)
    return min_training_samples_needed

# Main plotting function for multiple consecutive iterations
def plot_min_training_samples_multirun(al_strategy, casestudy, custom_titles, custom_legend, stability_threshold, consecutive_iters_list):
    cm = 1/2.54  # centimeters in inches
    fig, axs = plt.subplots(len(consecutive_iters_list), len(casestudy), figsize=(35*cm, 25*cm), sharex=True, sharey=True)
    fig.suptitle(rf'Number of training samples to reach $\epsilon$ < {stability_threshold:.1%}', y=0.92)
    
    data_per_case = {key: {} for key in consecutive_iters_list}
    # Plot for each row of consecutive iterations
    for row, consecutive_iterations in enumerate(consecutive_iters_list):
        min_training_samples_needed = calculate_min_training_samples(casestudy, al_strategy, stability_threshold, consecutive_iterations)

        for i, case in enumerate(casestudy):
            ax = axs[row, i]
            if row == 0:
                ax.set_title(custom_titles[i])

            if i == 0:
                ax.set_ylabel(f"{consecutive_iterations} iterations", fontsize=12)

            data = [min_training_samples_needed[case][strategy] for strategy in al_strategy]

            data_per_case[consecutive_iterations][case] = data
                # Reverse the order of data and strategies data.reverse()
            data.reverse() 
            strategies_reversed = al_strategy[::-1]

            bplot = ax.boxplot(
                data, vert=False, patch_artist=True, tick_labels=strategies_reversed, meanline=True, showmeans=True, widths=0.6,
                boxprops=dict(linewidth=1.0),
                medianprops=dict(linewidth=1.3, color=median_color, ls='--'),
                meanprops=dict(linewidth=1.3, color=mean_color, ls='-'), 
                whis = [2.5,97.5]
            )

            # Apply colors to each box based on strategy
            for patch, strategy in zip(bplot['boxes'], strategies_reversed):
                # patch.set_facecolor(strategy_colors[al_strategy[custom_legend.index(strategy)]])
                patch.set_facecolor(strategy_colors[strategy])

            ax.set_yticks([])
            ax.grid(True, which="both", linewidth=0.2, alpha=0.5)

    # Set a single x-axis label on the last row of plots
    for ax in axs[-1, :]:
        ax.set_xlabel("Training samples")
    
    # common_xlim = (0, max(all_data_values) + 10)
    common_xlim = (0, 200)

    for ax in axs.flat:
        ax.set_xlim(common_xlim)

    # Create custom handles for the strategy colors, median, and mean lines
    handles = [
        plt.Line2D([0], [0], color=color, marker='s', markersize=10, linestyle='', label=label)
        for label, color in zip(custom_legend, strategy_colors.values())
    ]
    handles.append(plt.Line2D([0], [1], color=median_color, linestyle='--', linewidth=1.0, label='Median'))
    handles.append(plt.Line2D([0], [1], color=mean_color, linestyle='-', linewidth=1.0, label='Mean'))

    # Add a single legend at the bottom of the figure
    fig.legend(handles=handles, title="Strategy and Statistics", loc="lower center", ncol=len(al_strategy) + 2)
    plt.tight_layout(rect=[0, 0.05, 1, 0.93])
    plt.show()
    return data_per_case, fig

In [ ]:
# Example usage:
custom_titles = [rf'Four branch, $k=6$', rf'Four branch, $k=7$', 'Hat', 'Himmelblau']
custom_legend = [rf'MOO Reliability', rf'Knee', rf'Compromised', rf'EFF', rf'$U$']
strategy_colors = {'mo_reliability': '#b3cde3', 
                   'knee': '#ccebc5', 
                   'compromise': '#decbe4',
                   'eff': '#ffffb3',
                   'u': '#fed9a6'}

median_color='#e41a1c'
mean_color='k'
threshold = 0.002
iterations = [2, 3, 4]
data_per_case_2, figure = plot_min_training_samples_multirun(al_strategy, casestudy, custom_titles, custom_legend, stability_threshold=threshold, consecutive_iters_list=iterations)

In [ ]:
# figure.savefig(f'stab_{threshold}.pdf',bbox_inches='tight')

In [ ]:
def plot_strategy_comparison_dual(data_per_case1, data_per_case2, strategies, consecutive_iterations_values):
    """
    Generates a figure with two heatmap comparison matrices of active learning strategies.

    Parameters:
    - data_per_case1: dict
        First dataset for the comparison matrix (e.g., for one stability threshold).
    - data_per_case2: dict
        Second dataset for the comparison matrix (e.g., for another stability threshold).
    - strategies: list
        List of strategy names.
    - consecutive_iterations_values: list
        List of consecutive iteration values considered.

    Outputs:
    - Displays and returns the matplotlib figure.
    """
    # Function to compute the total accumulated differences per strategy
    def compute_total_scores(data_per_case):
        total_scores_per_strategy = {strategy: 0 for strategy in strategies}

        for ci in consecutive_iterations_values:
            if ci not in data_per_case:
                continue

            for case in data_per_case[ci]:
                data_lists = data_per_case[ci][case]
                strategy_data = {}
                for idx, strategy in enumerate(strategies):
                    data_list = data_lists[idx]
                    strategy_data[strategy] = data_list

                medians = {}
                stddevs = {}
                scores = {}

                for strategy in strategies:
                    data_list = strategy_data[strategy]
                    data_clean = [200 if np.isnan(x) else x for x in data_list]
                    data_array = np.array(data_clean)
                    median = np.median(data_array)
                    std = np.std(data_array, ddof=1)
                    medians[strategy] = median
                    stddevs[strategy] = std

                median_values = list(medians.values())
                stddev_values = list(stddevs.values())
                median_min = min(median_values)
                median_max = max(median_values)
                stddev_min = min(stddev_values)
                stddev_max = max(stddev_values)

                median_range = median_max - median_min if median_max != median_min else 1e-6
                stddev_range = stddev_max - stddev_min if stddev_max != stddev_min else 1e-6

                case_scores = {}
                for strategy in strategies:
                    median_score = (medians[strategy] - median_min) / median_range
                    stddev_score = (stddevs[strategy] - stddev_min) / stddev_range
                    combined_score = median_score + stddev_score
                    case_scores[strategy] = combined_score

                winner_strategy = min(case_scores, key=case_scores.get)
                winner_score = case_scores[winner_strategy]

                for strategy in strategies:
                    diff = - case_scores[strategy] + winner_score
                    total_scores_per_strategy[strategy] += diff

        # Build comparison matrix
        comparison_matrix = np.zeros((len(strategies), len(strategies)))
        for i, strategy_i in enumerate(strategies):
            for j, strategy_j in enumerate(strategies):
                comparison_matrix[i, j] = total_scores_per_strategy[strategy_i] - total_scores_per_strategy[strategy_j]

        # Take absolute values
        # comparison_matrix_abs = np.abs(comparison_matrix)
        comparison_matrix_abs = comparison_matrix
        return comparison_matrix_abs, total_scores_per_strategy

    # Compute matrices and total differences for both datasets
    matrix1, total_differences1 = compute_total_scores(data_per_case1)
    matrix2, total_differences2 = compute_total_scores(data_per_case2)

    # Prepare labels with total accumulated differences
    custom_legend = [rf'$U$', rf'EFF', rf'Compromised', rf'Knee', rf'MOO Reliability']

    strategy_labels1 = [f"{custom}\n({total_differences1[strategy]:.2f})" for strategy, custom in zip(strategies, custom_legend)]
    strategy_labels2 = [f"{custom}\n({total_differences2[strategy]:.2f})" for strategy, custom in zip(strategies, custom_legend)]

    # Determine the combined maximum value for the colorbar
    vmax = max(matrix1.max(), matrix2.max())
    vmin = min(matrix1.min(), matrix2.min())

    norm = colors.Normalize(vmin=vmin, vmax=vmax)

    # Create subplots
    # Plotting the results
    cm = 1/2.54  # centimeters in inches
    plt.rcParams['font.family'] = 'times new roman'
    plt.rcParams['font.size'] = 15
    # plt.rcParams['text.usetex'] = False
    plt.rcParams['mathtext.fontset'] = 'stix'

    fig, axes = plt.subplots(1, 2, figsize=(36*cm, 18*cm), sharey=True)

    # Plot first heatmap
    sns.heatmap(matrix2, annot=True, fmt=".2f", ax=axes[0], cmap="YlGnBu", norm=norm,
                xticklabels=strategy_labels2, yticklabels=custom_legend, cbar=False, square=True)
    axes[0].set_title(rf"Accumulated score for $\epsilon$ < {0.002:.1%}")
    axes[0].set_xlabel("Total accumulated difference")
    # axes[0].set_ylabel("Strategy")

    # Plot second heatmap
    sns.heatmap(matrix1, annot=True, fmt=".2f", ax=axes[1], cmap="YlGnBu", norm=norm,
                xticklabels=strategy_labels1, yticklabels=custom_legend, cbar=False , square=True)
    axes[1].set_title(rf"Accumulated score for $\epsilon$ < {0.001:.1%}")
    axes[1].set_xlabel("Total accumulated difference")
    # axes[1].set_ylabel("")

    # Adjust layout to make room for colorbar
    plt.tight_layout(rect=[0, 0, 0.9, 1])  # leave space on the right for colorbar

    # Add a single colorbar
    cbar_ax = fig.add_axes([0.89, 0.15, 0.02, 0.78])  # [left, bottom, width, height]
    sm = plt.cm.ScalarMappable(cmap="YlGnBu", norm=norm)
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax)
    # cbar_ax.set_title('Difference')

    plt.show()
    return fig

In [ ]:
# Example usage:
# data_per_case1 and data_per_case2 should be your actual data dictionaries for the two datasets.
strategies = ['u', 'eff', 'compromise', 'knee', 'mo_reliability']
fig = plot_strategy_comparison_dual(data_per_case_1, data_per_case_2, strategies, iterations)
fig.savefig('dual_strategy_comparison_matrix.pdf', bbox_inches='tight')

In [ ]:
# Example usage:
# data_per_case1 and data_per_case2 should be your actual data dictionaries for the two datasets.
strategies = ['u', 'eff', 'compromise', 'knee', 'mo_reliability']
fig = plot_strategy_comparison_dual(data_per_case_1, data_per_case_2, strategies, iterations)
fig.savefig('dual_strategy_comparison_matrix.pdf', bbox_inches='tight')

In [ ]:
threshold = 0.002
figure = plot_min_training_samples_multirun(al_strategy, casestudy, custom_titles, custom_legend, stability_threshold=threshold, consecutive_iters_list=iterations)

In [ ]:
figure.savefig(f'stab_{threshold}.pdf',bbox_inches='tight')

In [ ]:
iterations = [2, 3, 4]
data_per_case = {key: [] for key in iterations}

In [ ]:
data_per_case

In [ ]:
strategies = ['MO Reliability', 'Knee', 'Compromise', 'EFF', 'U']
n_strategies = len(strategies)
comparison_matrix = np.zeros((n_strategies, n_strategies), dtype=int)

# For each stability threshold and consecutive iteration setting
stability_thresholds = [0.002]  # Example thresholds
consecutive_iterations_list = [2, 3, 4]  # Example iterations

for stability_threshold in stability_thresholds:
    for consecutive_iterations in consecutive_iterations_list:
        # Assuming min_training_samples_needed is updated accordingly
        for row, consecutive_iterations in enumerate(iterations):
            min_training_samples_needed = calculate_min_training_samples(casestudy, al_strategy, stability_threshold, consecutive_iterations)
            for case in casestudy:
                medians = []
                for strategy in al_strategy:
                    data = min_training_samples_needed[case][strategy]  # Data under current criteria
                    median_value = np.nanmedian(data)
                    medians.append(median_value)
                
                # Perform pairwise comparisons
                for i in range(n_strategies):
                    for j in range(n_strategies):
                        if i != j:
                            if medians[i] < medians[j]:
                                comparison_matrix[i][j] += 1
                            elif medians[i] > medians[j]:
                                comparison_matrix[j][i] += 1
                            # For ties, you can choose to increment both or neither


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
ax = sns.heatmap(comparison_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=strategies, yticklabels=strategies)
plt.title('Pairwise Strategy Performance Comparison')
plt.xlabel('Strategy Outperformed')
plt.ylabel('Strategy')
plt.show()


In [ ]:
n_exp = 15
casestudy = ['four_branch_6', 'four_branch_7', 'hat', 'himmelblau']
al_strategy = ['u', 'eff', 'knee', 'compromise']
name_exp = list(range(1, n_exp+1))

# Initialize the nested dictionary to store results
results_dict = {case: {strategy: [] for strategy in al_strategy} for case in casestudy}

# Loop through each case study and strategy
for case in casestudy:
    for strategy in al_strategy:
        for exp_num in range(1, n_exp + 1):
            key = f"{case}_{strategy}_1_{exp_num}"
            # Check if key exists in output_results_dict
            if key in output_results_dict:
                # Access Pareto_metrics and store in the nested dictionary
                pareto_metrics = np.array(output_results_dict[key]['Pareto_metrics'])
                results_dict[case][strategy].append(pareto_metrics)

def plot_pareto_metrics(results_dict, casestudy, al_strategy):
    # Access the 15 experiments for the specified case study and strategy
    experiments_data = results_dict[casestudy][al_strategy]
    
    # Ensure there are 15 experiments
    if len(experiments_data) != 15:
        print("Warning: Expected 15 experiments but found", len(experiments_data))
    
    # Prepare the subplots
    fig, axes = plt.subplots(2, 1, figsize=(10, 6))
    fig.suptitle(f'Evolution of Pareto Metrics for {casestudy} - Strategy: {al_strategy}', fontsize=14)
    
    # Plot each experiment's metrics on separate subplots
    for exp_num, metrics in enumerate(experiments_data, start=1):
        # Plot first metric: 'distance to the boundary line'
        axes[0].plot(metrics[:, 0], label=f'Experiment {exp_num}', alpha=0.7)
        axes[0].set_title('Distance to the Boundary Line')
        axes[0].set_xlabel('Iteration')
        axes[0].set_ylabel('Distance')
        # axes[0].set_yscale('log')
        
        # Plot second metric: 'distance to extreme points'
        axes[1].plot(metrics[:, 1], label=f'Experiment {exp_num}', alpha=0.7)
        axes[1].set_title('Distance to Extreme Points')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Distance')
        # axes[1].set_yscale('log')
    
    # # Add legends to both subplots
    # axes[0].legend(loc='best', fontsize='small', ncol=3)
    # axes[1].legend(loc='best', fontsize='small', ncol=3)
    
    # Adjust layout for better readability
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
plot_pareto_metrics(results_dict, 'four_branch_6', 'u')

In [ ]:
plot_pareto_metrics(results_dict, 'four_branch_6', 'eff')

In [ ]:
plot_pareto_metrics(results_dict, 'four_branch_6', 'knee')

In [ ]:
plot_pareto_metrics(results_dict, 'four_branch_6', 'compromise')

In [ ]:
plt.plot(pareto_metrics[:, 0])

In [ ]:
# Plot for Distance to Extreme Points (second column in Pareto_metrics)
plot_metric("Average distance to extreme points", data_col=1, ylabel="Average distance to extreme points")

Reading Pf output files

In [ ]:
n_exp = 10
casestudy = ['four_branch_6', 'four_branch_7', 'hat', 'himmelblau']
name_exp = list(range(1, n_exp+1))
# Base results directory
base_results_dir = '/Users/jonathan/Documents/MOAL/Experiments/AL_StructuralReliability/results_pf_1e8'

# Dictionaries to store results
output_results_dict = {}
config_results_dict = {}

for case in casestudy:
    for exp_num in name_exp:
        # Construct the directory pattern
        dir_pattern = os.path.join(
            base_results_dir, case, f"{exp_num}_*"
        )
        
        # Find matching directories
        matching_dirs = glob.glob(dir_pattern)
        
        if matching_dirs:
            # Define the unique key for this experiment setting
            key = f"{case}_{exp_num}"
            
            # Load output.json if it exists
            output_json_path = os.path.join(matching_dirs[0], 'output.json')
            if os.path.isfile(output_json_path):
                with open(output_json_path, 'r') as f:
                    output_data = json.load(f)
                output_results_dict[key] = output_data
            else:
                print(f"No output.json found for {key}")
            
            # Load config.json if it exists
            config_json_path = os.path.join(matching_dirs[0], 'config.json')
            if os.path.isfile(config_json_path):
                with open(config_json_path, 'r') as f:
                    config_data = json.load(f)
                config_results_dict[key] = config_data
            else:
                print(f"No config.json found for {key}")
        else:
            print(f"No directory found for {case}, exp {exp_num}")